# Project 13 · PubMedBERT Baseline for Disease Named Entity Recognition

This notebook fine-tunes Microsoft's **abstracts-only PubMedBERT**, now called BiomedBERT, by default. It uses the same NCBI Disease splits and first-subword label alignment as the team's baseline notebook. Training runs for three epochs, with validation metrics computed after each epoch. Test evaluation uses the weights from epoch 3 to match the team's original evaluation procedure.

The test F1 score of **0.8528** in the teammate's attachment comes from the general-purpose `bert-base-cased` model. To compare PubMedBERT and BioBERT, measure both models using the same procedure.

[PubMedBERT model card](https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract) · [BioBERT model card](https://huggingface.co/dmis-lab/biobert-base-cased-v1.2) · [NCBI Disease dataset card](https://huggingface.co/datasets/ncbi/ncbi_disease)

## 1. Set up the environment

Run the cells from top to bottom in Jupyter or Colab with internet access and a GPU. In Colab, choose a GPU under Runtime → Change runtime type. If prompted to restart the kernel after installing dependencies, restart it and continue from the configuration cell. Run the default PubMedBERT experiment first, then set `MODELS_TO_RUN` to `["BioBERT"]` and rerun from the configuration cell. You can also run both models together. Each run saves its outputs under `ner_baseline_runs/` in the current working directory. Download or save that directory to persistent storage before ending your Colab session.

In [1]:
%pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets==3.6.0" "accelerate>=1.1,<2" "seqeval==1.2.2"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.4 MB/s  0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=42664bba9c12d2b2a7862c85b2df05d8f7094688cfbe0ddfe3082af516173c3d
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1


In [2]:
import csv
import gc
import hashlib
import json
import tempfile
from pathlib import Path
from urllib.request import urlopen

import datasets
import numpy as np
import torch
import transformers
from datasets import Dataset, DatasetDict
from IPython.display import Markdown, display
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,
    Trainer, TrainingArguments, set_seed,
)

MODELS_TO_RUN = ["PubMedBERT"]
MODEL_IDS = {
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
    "BioBERT": "dmis-lab/biobert-base-cased-v1.2",
}
MODEL_REVISIONS = {
    "PubMedBERT": "d673b8835373c6fa116d6d8006b33d48734e305d",
    "BioBERT": "67c9c25b46986521ca33df05d8540da1210b3256",
}
OUTPUT_ROOT = Path("ner_baseline_runs")
SEED = 42
MAX_LENGTH = 512
LABELS = ["O", "B-Disease", "I-Disease"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
print("CUDA:", torch.cuda.is_available(), "| PyTorch:", torch.__version__)

CUDA: True | PyTorch: 2.14.1+cu130


## 2. Load and inspect the dataset

Read the CoNLL files used by the Hugging Face loading script. Follow its sentence splitting and label definitions, including the empty record that the original loader produces at the end of each file. Check the sample count for each split and record the SHA-256 hash of each raw file so later experiments can verify that they used the same data.

In [3]:
DATA_URL = "https://raw.githubusercontent.com/spyysalo/ncbi-disease/master/conll/"
SPLIT_FILES = {"train": "train.tsv", "validation": "devel.tsv", "test": "test.tsv"}
EXPECTED_SIZES = {"train": 5433, "validation": 924, "test": 941}

def parse_conll(content):
    sentences = []
    tokens, tags = [], []
    for line in content.splitlines():
        if not line.strip():
            if tokens:
                sentences.append({"tokens": tokens, "ner_tags": tags})
                tokens, tags = [], []
            continue
        token, tag = line.split("\t")
        tokens.append(token)
        tags.append(LABEL_TO_ID[tag.strip()])
    sentences.append({"tokens": tokens, "ner_tags": tags})
    return sentences

dataset = DatasetDict()
data_hashes = {}
for split, filename in SPLIT_FILES.items():
    with urlopen(DATA_URL + filename, timeout=60) as response:
        raw = response.read()
    data_hashes[split] = hashlib.sha256(raw).hexdigest()
    rows = parse_conll(raw.decode("utf-8"))
    assert len(rows) == EXPECTED_SIZES[split], (split, len(rows))
    dataset[split] = Dataset.from_list(rows)
print({split: len(rows) for split, rows in dataset.items()})

{'train': 5433, 'validation': 924, 'test': 941}


## 3. Tokenize and align labels

Only the first subword of each original word contributes to training and evaluation. Other subwords, special tokens, and padding positions use `-100` and are ignored. Precision, recall, and F1 use the team's default entity-level seqeval rules. Accuracy measures word-level label accuracy. Each model uses its own tokenizer, and the notebook checks that tokenization preserves a label for every original word.

In [4]:
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predicted_ids = np.argmax(logits, axis=-1)
    predictions, references = [], []
    for pred_row, label_row in zip(predicted_ids, labels):
        kept = label_row != -100
        predictions.append([LABELS[index] for index in pred_row[kept]])
        references.append([LABELS[index] for index in label_row[kept]])
    return {
        "precision": precision_score(references, predictions, zero_division=0),
        "recall": recall_score(references, predictions, zero_division=0),
        "f1": f1_score(references, predictions, zero_division=0),
        "accuracy": accuracy_score(references, predictions),
    }

## 4. Train, validate, and evaluate

Use a learning rate of 2e-5, a per-device batch size of 8, three epochs, weight decay of 0.01, and a fixed random seed of 42. BF16 mixed precision is enabled on CUDA GPUs that support it. Both models use the same settings. The training log records validation results after each epoch. Each run has its own directory and saves the final model weights, tokenizer, training configuration, and metrics. Only the final model is saved to reduce disk usage.

In [5]:
def run_baseline(name):
    model_id = MODEL_IDS[name]
    revision = MODEL_REVISIONS[name]
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, use_fast=True, do_lower_case=(name == "PubMedBERT")
    )
    encoded = dataset.map(
        tokenize_and_align_labels,
        batched=True,
        fn_kwargs={"tokenizer": tokenizer},
        remove_columns=dataset["train"].column_names,
    )
    for split, rows in encoded.items():
        for index, (aligned, original_tags) in enumerate(zip(rows["labels"], dataset[split]["ner_tags"])):
            kept = [tag for tag in aligned if tag != -100]
            if kept != original_tags:
                raise ValueError(f"{name}: Missing or misaligned labels in {split} record {index}")

    model = AutoModelForTokenClassification.from_pretrained(
        model_id, revision=revision, num_labels=len(LABELS), label2id=LABEL_TO_ID,
        id2label=dict(enumerate(LABELS)),
    )
    model_root = OUTPUT_ROOT / name.lower()
    model_root.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=model_root))
    args = TrainingArguments(
        output_dir=str(run_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        seed=SEED,
        data_seed=SEED,
        optim="adamw_torch",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded["train"],
        eval_dataset=encoded["validation"],
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    test = trainer.evaluate(encoded["test"], metric_key_prefix="test")
    trainer.save_model(str(run_dir / "model"))
    trainer.state.save_to_json(str(run_dir / "trainer_state.json"))
    result = {
        "model": name,
        "checkpoint": model_id,
        "model_revision": revision,
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "data_sha256": data_hashes,
        "versions": {"torch": str(torch.__version__), "datasets": datasets.__version__,
                     "transformers": transformers.__version__},
        "training_args": args.to_dict(),
        "test": test,
    }
    (run_dir / "results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"{name} saved to {run_dir.resolve()}")

for model_name in MODELS_TO_RUN:
    run_baseline(model_name)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

... (2 outputs left out)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.031300,0.052938,0.837905,0.853875,0.845815,0.985523
2,0.024500,0.045324,0.830952,0.886912,0.858021,0.986733
3,0.026300,0.059411,0.853659,0.889454,0.871189,0.987234


PubMedBERT saved to /content/ner_baseline_runs/pubmedbert/run-fgk116p1


## 5. Generate the test-set comparison table

Load the most recent complete run for each model that matches the current data files. Scores in the table are shown on a 0–1 scale. The reference scores for general-domain BERT come from the teammate's attached results. Once test results are available for both BioBERT and PubMedBERT, the notebook reports the F1 difference in percentage points.

In [6]:
columns = ["model", "precision", "recall", "f1", "accuracy"]
rows = [{
    "model": "BERT-base-cased (teammate's results)",
    "precision": 0.826171875,
    "recall": 0.88125,
    "f1": 0.8528225806451613,
    "accuracy": 0.9838755766012165,
}]
for name in ("BioBERT", "PubMedBERT"):
    paths = (OUTPUT_ROOT / name.lower()).glob("run-*/results.json")
    row = {"model": name}
    for path in sorted(paths, key=lambda item: item.stat().st_mtime, reverse=True):
        result = json.loads(path.read_text(encoding="utf-8"))
        if (
            result["data_sha256"] == data_hashes
            and result["checkpoint"] == MODEL_IDS[name]
            and result["model_revision"] == MODEL_REVISIONS[name]
        ):
            row.update({key: result["test"][f"test_{key}"] for key in columns[1:]})
            break
    rows.append(row)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with (OUTPUT_ROOT / "test_comparison.csv").open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

table = ["| Model | Precision | Recall | F1 | Accuracy |", "|---|---:|---:|---:|---:|"]
for row in rows:
    values = [f"{row[key]:.4f}" if key in row else "Training pending" for key in columns[1:]]
    table.append("| " + " | ".join([row["model"], *values]) + " |")
display(Markdown("\n".join(table)))
if all("f1" in row for row in rows[1:]):
    print(f"PubMedBERT − BioBERT test F1: {100 * (rows[2]['f1'] - rows[1]['f1']):+.2f} percentage points")

| Model | Precision | Recall | F1 | Accuracy |
|---|---:|---:|---:|---:|
| BERT-base-cased (teammate's results) | 0.8262 | 0.8812 | 0.8528 | 0.9839 |
| BioBERT | Training pending | Training pending | Training pending | Training pending |
| PubMedBERT | 0.8593 | 0.9031 | 0.8807 | 0.9860 |